# Lesson 17 — Data Cleaning & Missing Values with Pandas

## Quality Engineering / Manufacturing Data Analysis

**Goal:** Learn how to detect, understand, clean, and safely handle missing, duplicate, inconsistent, and invalid manufacturing-quality data before statistical analysis or machine learning.

### Learning outcomes
- Detect missing values with `isna()` / `isnull()`
- Measure missingness by column and row
- Handle missing values using `dropna()` and `fillna()`
- Use mean, median, mode and group-wise imputation
- Detect duplicate records
- Standardize text categories
- Detect invalid numeric values and impossible measurements
- Understand why blindly filling missing data can be dangerous in QA
- Build a reusable data-cleaning workflow


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## 1. Create a deliberately imperfect QA dataset

Real manufacturing data often contains missing readings, inconsistent category names, duplicate rows, and impossible measurements.

In [2]:
df = pd.DataFrame({
    'Date': pd.to_datetime([
        '2026-09-01','2026-09-01','2026-09-01','2026-09-02','2026-09-02',
        '2026-09-02','2026-09-03','2026-09-03','2026-09-03','2026-09-04',
        '2026-09-04','2026-09-04','2026-09-05','2026-09-05','2026-09-05'
    ]),
    'Line': ['L1','L1','L2','L2','L1','L2','L1','L1','L2','L2','L1','L2','L1','L2','L1'],
    'Shift': ['A','B','A','B','A','B','A','B','A','B','A','B','A','B','A'],
    'Model': ['M1','M1','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2','M1','M2','M1'],
    'Inspected': [980, 950, 1020, 990, np.nan, 970, 1030, 1000, 1020, 995, 990, 1000, 1015, 1035, 1010],
    'Defects': [18, 25, 12, np.nan, 15, 28, 17, 31, 14, 26, 22, 27, 16, 19, 32],
    'Temperature_C': [24.2, 24.8, np.nan, 25.1, 24.5, 25.0, 24.3, 24.7, 25.2, 24.9, 24.4, np.nan, 24.6, 25.1, 24.5],
    'Operator': ['Amit','Amit','Ravi','Ravi','Amit','Ravi','Amit','AMIT ','Ravi','Ravi','Amit','Ravi','Amit','Ravi','Amit'],
    'Status': ['OK','OK','OK','NG','OK','NG','OK','ok','OK','NG','OK','NG','OK','OK','OK']
})

# Add a deliberate duplicate of the first record.
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

# Add deliberate data-entry errors.
df.loc[7, 'Temperature_C'] = 250
df.loc[12, 'Defects'] = -4

df

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status
0,2026-09-01,L1,A,M1,980.000,18.000,24.200,Amit,OK
1,2026-09-01,L1,B,M1,950.000,25.000,24.800,Amit,OK
2,2026-09-01,L2,A,M1,1020.000,12.000,NaN,Ravi,OK
3,2026-09-02,L2,B,M2,990.000,NaN,25.100,Ravi,NG
4,2026-09-02,L1,A,M1,NaN,15.000,24.500,Amit,OK
5,2026-09-02,L2,B,M2,970.000,28.000,25.000,Ravi,NG
6,2026-09-03,L1,A,M1,1030.000,17.000,24.300,Amit,OK
7,2026-09-03,L1,B,M2,1000.000,31.000,250.000,AMIT,ok
8,2026-09-03,L2,A,M1,1020.000,14.000,25.200,Ravi,OK
9,2026-09-04,L2,B,M2,995.000,26.000,24.900,Ravi,NG


## 2. First step: inspect the dataset

In [3]:
df.shape, df.dtypes

((16, 9),
 Date             datetime64[ns]
 Line                     object
 Shift                    object
 Model                    object
 Inspected               float64
 Defects                 float64
 Temperature_C           float64
 Operator                 object
 Status                   object
 dtype: object)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           16 non-null     datetime64[ns]
 1   Line           16 non-null     object        
 2   Shift          16 non-null     object        
 3   Model          16 non-null     object        
 4   Inspected      15 non-null     float64       
 5   Defects        15 non-null     float64       
 6   Temperature_C  14 non-null     float64       
 7   Operator       16 non-null     object        
 8   Status         16 non-null     object        
dtypes: datetime64[ns](1), float64(3), object(5)
memory usage: 1.3+ KB


## 3. Detect missing values

In [5]:
df.isna().sum()

Date             0
Line             0
Shift            0
Model            0
Inspected        1
Defects          1
Temperature_C    2
Operator         0
Status           0
dtype: int64

In [6]:
missing_pct = df.isna().mean().mul(100).sort_values(ascending=False)
missing_pct

Temperature_C   12.500
Defects          6.250
Inspected        6.250
Date             0.000
Line             0.000
Model            0.000
Shift            0.000
Operator         0.000
Status           0.000
dtype: float64

### QA interpretation

Missingness is not automatically an error. A missing measurement may mean:
- the sensor was unavailable,
- the inspection was skipped,
- the value was not applicable,
- data transfer failed,
- or the operator failed to record it.

Before imputing a value, understand **why it is missing**.

## 4. Find rows containing missing values

In [7]:
df[df.isna().any(axis=1)]

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status
2,2026-09-01,L2,A,M1,1020.000,12.000,NaN,Ravi,OK
3,2026-09-02,L2,B,M2,990.000,NaN,25.100,Ravi,NG
4,2026-09-02,L1,A,M1,NaN,15.000,24.500,Amit,OK
11,2026-09-04,L2,B,M2,1000.000,27.000,NaN,Ravi,NG


## 5. Find rows where a specific column is missing

In [8]:
df[df['Temperature_C'].isna()]

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status
2,2026-09-01,L2,A,M1,1020.000,12.000,NaN,Ravi,OK
11,2026-09-04,L2,B,M2,1000.000,27.000,NaN,Ravi,NG


## 6. `dropna()` — remove incomplete records

In [9]:
complete_rows = df.dropna()
complete_rows.shape

(12, 9)

Dropping every row with any missing value can remove too much useful production data. Use it only when the analysis requires complete cases.

In [10]:
# Drop rows only when Inspected is missing
df_inspected_available = df.dropna(subset=['Inspected'])
df_inspected_available.shape

(15, 9)

## 7. `fillna()` — simple imputation

In [11]:
df_simple = df.copy()
df_simple['Temperature_C'] = df_simple['Temperature_C'].fillna(df_simple['Temperature_C'].median())

df_simple[['Temperature_C']].isna().sum()

Temperature_C    0
dtype: int64

### Mean vs Median

- **Mean:** sensitive to outliers.
- **Median:** more robust when the distribution is skewed or contains outliers.

For process measurements, the choice should be justified by process knowledge and the analysis objective.

## 8. Mode for categorical data

In [12]:
df_mode = df.copy()
mode_status = df_mode['Status'].mode()[0]
df_mode['Status'] = df_mode['Status'].fillna(mode_status)
df_mode['Status'].value_counts()

Status
OK    11
NG     4
ok     1
Name: count, dtype: int64

## 9. Forward fill and backward fill

In [13]:
temp_demo = df[['Date', 'Temperature_C']].copy()
temp_demo['ffill'] = temp_demo['Temperature_C'].ffill()
temp_demo['bfill'] = temp_demo['Temperature_C'].bfill()
temp_demo

,Date,Temperature_C,ffill,bfill
0,2026-09-01,24.200,24.200,24.200
1,2026-09-01,24.800,24.800,24.800
2,2026-09-01,NaN,24.800,25.100
3,2026-09-02,25.100,25.100,25.100
4,2026-09-02,24.500,24.500,24.500
5,2026-09-02,25.000,25.000,25.000
6,2026-09-03,24.300,24.300,24.300
7,2026-09-03,250.000,250.000,250.000
8,2026-09-03,25.200,25.200,25.200
9,2026-09-04,24.900,24.900,24.900


**Caution:** Forward/backward filling is not automatically valid for process data. It assumes the previous or next observation is a reasonable representation of the missing point.

## 10. Group-wise imputation

If different production lines operate at different normal temperatures, a global median may be inappropriate. Use a line-specific baseline instead.

In [14]:
df_group = df.copy()
df_group['Temperature_C'] = df_group.groupby('Line')['Temperature_C'].transform(
    lambda s: s.fillna(s.median())
)

df_group[['Line', 'Temperature_C']].isna().sum()

Line             0
Temperature_C    0
dtype: int64

## 11. Detect duplicate records

In [15]:
df.duplicated().sum()

np.int64(1)

In [16]:
df[df.duplicated(keep=False)]

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status
0,2026-09-01,L1,A,M1,980.000,18.000,24.200,Amit,OK
15,2026-09-01,L1,A,M1,980.000,18.000,24.200,Amit,OK


## 12. Remove duplicates

In [17]:
df_dedup = df.drop_duplicates().copy()
df_dedup.shape

(15, 9)

### QA warning

Do not delete duplicates blindly. Two rows can legitimately have the same values. First define what makes a record unique, such as:
`Date + Line + Shift + Model + Batch_ID`.

## 13. Standardize text categories

In [18]:
df_clean = df_dedup.copy()

df_clean['Operator'] = df_clean['Operator'].str.strip().str.title()
df_clean['Status'] = df_clean['Status'].str.strip().str.upper()

print(df_clean['Operator'].unique())
print(df_clean['Status'].unique())

['Amit' 'Ravi']
['OK' 'NG']


## 14. Detect invalid numeric values

In [19]:
# Defects cannot be negative.
df_clean[df_clean['Defects'] < 0]

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status
12,2026-09-05,L1,A,M1,1015.000,-4.000,24.600,Amit,OK


In [20]:
# Temperature values outside a practical engineering range should be investigated.
df_clean[(df_clean['Temperature_C'] < 15) | (df_clean['Temperature_C'] > 40)]

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status
7,2026-09-03,L1,B,M2,1000.000,31.000,250.000,Amit,OK


The correct action for an invalid value is not always to replace it. Depending on the root cause, it may be:
- corrected from the source system,
- marked invalid,
- excluded from a particular analysis,
- or retained for audit purposes.

## 15. Create validation flags instead of silently changing data

In [21]:
df_clean['Defects_Valid'] = df_clean['Defects'].between(0, df_clean['Inspected'], inclusive='both')
df_clean['Temperature_Valid'] = df_clean['Temperature_C'].between(15, 40, inclusive='both')

df_clean[['Defects', 'Inspected', 'Defects_Valid', 'Temperature_C', 'Temperature_Valid']]

,Defects,Inspected,Defects_Valid,Temperature_C,Temperature_Valid
0,18.000,980.000,True,24.200,True
1,25.000,950.000,True,24.800,True
2,12.000,1020.000,True,NaN,False
3,NaN,990.000,False,25.100,True
4,15.000,NaN,False,24.500,True
5,28.000,970.000,True,25.000,True
6,17.000,1030.000,True,24.300,True
7,31.000,1000.000,True,250.000,False
8,14.000,1020.000,True,25.200,True
9,26.000,995.000,True,24.900,True


## 16. Check logical relationships

In [22]:
# Defects should not exceed inspected quantity.
invalid_defect_logic = df_clean[
    (df_clean['Defects'] < 0) |
    (df_clean['Inspected'] < 0) |
    (df_clean['Defects'] > df_clean['Inspected'])
]

invalid_defect_logic

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status,Defects_Valid,Temperature_Valid
12,2026-09-05,L1,A,M1,1015.000,-4.000,24.600,Amit,OK,False,True


## 17. Handle invalid values explicitly

In [23]:
df_valid = df_clean.copy()

# Replace invalid defect counts with NaN so they are not silently treated as real measurements.
df_valid.loc[~df_valid['Defects_Valid'], 'Defects'] = np.nan

# Replace invalid temperature readings with NaN.
df_valid.loc[~df_valid['Temperature_Valid'], 'Temperature_C'] = np.nan

df_valid[['Defects', 'Temperature_C']].isna().sum()

Defects          3
Temperature_C    3
dtype: int64

## 18. Build a reusable cleaning function

In [24]:
def clean_quality_data(data):
    out = data.copy()

    # Standardize text
    out['Operator'] = out['Operator'].str.strip().str.title()
    out['Status'] = out['Status'].str.strip().str.upper()

    # Validate numeric relationships
    out['Defects_Valid'] = out['Defects'].between(0, out['Inspected'], inclusive='both')
    out['Temperature_Valid'] = out['Temperature_C'].between(15, 40, inclusive='both')

    # Preserve invalid-data flags, then set invalid measurements to NaN
    out.loc[~out['Defects_Valid'], 'Defects'] = np.nan
    out.loc[~out['Temperature_Valid'], 'Temperature_C'] = np.nan

    # Remove exact duplicate records
    out = out.drop_duplicates().reset_index(drop=True)

    return out

cleaned = clean_quality_data(df)
cleaned.head()

,Date,Line,Shift,Model,Inspected,Defects,Temperature_C,Operator,Status,Defects_Valid,Temperature_Valid
0,2026-09-01,L1,A,M1,980.000,18.000,24.200,Amit,OK,True,True
1,2026-09-01,L1,B,M1,950.000,25.000,24.800,Amit,OK,True,True
2,2026-09-01,L2,A,M1,1020.000,12.000,NaN,Ravi,OK,True,False
3,2026-09-02,L2,B,M2,990.000,NaN,25.100,Ravi,NG,False,True
4,2026-09-02,L1,A,M1,NaN,NaN,24.500,Amit,OK,False,True


## 19. Cleaning audit summary

In [25]:
audit = pd.DataFrame({
    'Rows': [len(df), len(cleaned)],
    'Missing_Inspected': [df['Inspected'].isna().sum(), cleaned['Inspected'].isna().sum()],
    'Missing_Defects': [df['Defects'].isna().sum(), cleaned['Defects'].isna().sum()],
    'Missing_Temperature': [df['Temperature_C'].isna().sum(), cleaned['Temperature_C'].isna().sum()],
    'Duplicate_Rows': [df.duplicated().sum(), cleaned.duplicated().sum()]
}, index=['Before', 'After'])

audit

,Rows,Missing_Inspected,Missing_Defects,Missing_Temperature,Duplicate_Rows
Before,16,1,1,2,1
After,15,1,3,3,0


## 20. Practice Exercises

### Exercise 1
Calculate missing-value percentage for every column.

### Exercise 2
Find all duplicate records.

### Exercise 3
Standardize the `Operator` column so `AMIT ` and `Amit` become the same category.

### Exercise 4
Find all negative defect values.

### Exercise 5
Find all temperature readings above 40°C.

### Exercise 6
Replace missing temperature values using the median temperature of the corresponding Line.

### Exercise 7
Create a `Data_Quality_Issue` column with values such as `Missing`, `Invalid`, `Duplicate`, or `OK`.

### Exercise 8
After cleaning, calculate total inspected quantity, total defects and weighted defect rate.

In [26]:
# Hints
# Exercise 1: df.isna().mean() * 100
# Exercise 2: df[df.duplicated(keep=False)]
# Exercise 3: df['Operator'].str.strip().str.title()
# Exercise 4: df[df['Defects'] < 0]
# Exercise 5: df[df['Temperature_C'] > 40]
# Exercise 6: df.groupby('Line')['Temperature_C'].transform(lambda s: s.fillna(s.median()))
# Exercise 7: use np.select() with logical conditions
# Exercise 8: total defects / total inspected * 100

# Mini Project — QA Data Cleaning Pipeline

Create a production-ready function that:

1. Standardizes categorical text.
2. Detects duplicates.
3. Validates Defects against Inspected.
4. Validates Temperature against engineering limits.
5. Preserves validation flags.
6. Handles missing values using an explicitly documented method.
7. Produces a before/after audit summary.
8. Returns a clean dataset ready for statistical analysis.

### QA mindset

**Data cleaning is part of measurement-system discipline.** Never change a measurement merely to make the dataset look clean. Preserve evidence of what was wrong, why it was changed, and what rule was applied.

# Key Takeaways

- `isna()` / `isnull()` → detect missing data.
- `dropna()` → remove incomplete records selectively.
- `fillna()` → impute missing values.
- `transform()` → perform group-wise imputation.
- `duplicated()` / `drop_duplicates()` → manage duplicate records.
- `str.strip()` / `str.upper()` / `str.title()` → standardize categories.
- Validation flags → identify bad measurements without hiding them.
- Domain rules → essential for identifying impossible manufacturing values.

**Next lesson:** Exploratory Data Analysis (EDA) — distributions, outliers, relationships, and quality-oriented visual investigation.